# Scraper Puntual de Catálogo de Productos

Notebook de extracción incremental en **CSV** (`scrap/data/catalogo_productos_raw.csv`, configurable vía `OUTPUT_CSV_REL` en `.env`) sin duplicados por `codigo`, extrayendo únicamente las 7 columnas estructurales del catálogo (sin datos comerciales ni precios).

**Requisitos previos:**
- Configurar las variables de entorno en `.env` (ver `.env.example`).
- Tener Google Chrome abierto con `--remote-debugging-port=9222` y sesión iniciada en el portal configurado en `TARGET_DOMAIN`.

## 1. Configuración y variables de entorno
Carga librerías, resuelve la raíz del proyecto e importa la configuración desde `.env` y los términos de búsqueda.

In [ ]:
import json
import os
import random
import time
import unicodedata
from pathlib import Path

import pandas as pd
import requests
import websocket
from dotenv import load_dotenv


_cwd = Path.cwd().resolve()
ROOT_DIR = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "scrap").is_dir() and (p / "etl").is_dir()),
    _cwd,
)
load_dotenv(ROOT_DIR / ".env")


CHROME_DEBUG_PORT = int(os.getenv("CHROME_DEBUG_PORT", "9222"))
CHROME_DEBUG_URL = f"http://127.0.0.1:{CHROME_DEBUG_PORT}/json"
TARGET_DOMAIN = os.getenv("TARGET_DOMAIN", "")
ORIGIN_URL = os.getenv("ORIGIN_URL", "")
REFERER_URL = os.getenv("REFERER_URL", "")
API_URL = os.getenv("API_URL", "")
OUTPUT_CSV = ROOT_DIR / os.getenv("OUTPUT_CSV_REL", "scrap/data/catalogo_productos_raw.csv")
PAGE_SIZE = 1000

TERMINOS_BUSQUEDA = [
    "*A", "*E", "*I", "*O", "*U"
]

## 2. Autenticación y normalización
Normaliza texto, extrae el token de sesión desde Chrome DevTools (`localStorage`) y arma los headers HTTP.

In [ ]:
def normalizar_texto(texto):
    if not texto:
        return ""
    texto_limpio = "".join(
        c for c in unicodedata.normalize("NFD", str(texto))
        if unicodedata.category(c) != "Mn"
    )
    return texto_limpio.strip().upper()


def get_local_storage(page_id):
    ws_url = f"ws://127.0.0.1:{CHROME_DEBUG_PORT}/devtools/page/{page_id}"
    try:
        ws = websocket.create_connection(ws_url, timeout=10)
        command = {
            "id": 1,
            "method": "Runtime.evaluate",
            "params": {
                "expression": """
                    (function() {
                        const items = {};
                        for (let i = 0; i < localStorage.length; i++) {
                            const key = localStorage.key(i);
                            items[key] = localStorage.getItem(key);
                        }
                        return items;
                    })()
                """,
                "returnByValue": True,
            },
        }
        ws.send(json.dumps(command))
        response = json.loads(ws.recv())
        ws.close()
        if "result" in response and "result" in response["result"]:
            return response["result"]["result"].get("value")
    except Exception as e:
        print(f"[ERROR] WebSocket al leer localStorage: {e}")
    return None


def get_auth_token(page_id):
    storage = get_local_storage(page_id)
    if not storage:
        print("[ERROR] No se pudo obtener el localStorage")
        return None

    for key in ["auth", "token", "auth_token", "access_token", "jwt", "authorization"]:
        if key in storage:
            valor = storage[key]
            try:
                datos_json = json.loads(valor)
                if isinstance(datos_json, dict) and "token" in datos_json:
                    return datos_json["token"]
            except Exception:
                pass
            return valor
    return None


def get_token():
    try:
        response = requests.get(CHROME_DEBUG_URL, timeout=5)
        response.raise_for_status()
        pages = response.json()
    except Exception as e:
        print(f"[ERROR] No se pudo conectar a Chrome en el puerto {CHROME_DEBUG_PORT}: {e}")
        return None

    target_page = next((p for p in pages if TARGET_DOMAIN and TARGET_DOMAIN in p.get("url", "")), None)
    if not target_page:
        print(f"[ERROR] No se encontró una pestaña abierta para el dominio configurado ({TARGET_DOMAIN})")
        return None

    token = get_auth_token(target_page["id"])
    if token:
        print("[OK] Token de autenticación obtenido correctamente")
    else:
        print("[ERROR] No se pudo extraer el token de localStorage")
    return token


def obtener_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json, text/plain, */*",
        "Origin": ORIGIN_URL,
        "Referer": REFERER_URL,
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    }

## 3. Persistencia CSV y consulta paginada
Lee códigos existentes para evitar duplicados, guarda nuevos registros en CSV y consulta la API por páginas.

In [ ]:
def cargar_codigos_existentes_csv(ruta_csv):
    codigos = set()
    if ruta_csv.exists():
        try:
            df_existente = pd.read_csv(ruta_csv, dtype={"codigo": str}, encoding="utf-8")
            if "codigo" in df_existente.columns:
                codigos = set(df_existente["codigo"].dropna().astype(str).str.strip())
            print(f"[INFO] Códigos previos cargados desde CSV: {len(codigos):,}")
        except Exception as e:
            print(f"[ERROR] Al leer CSV existente: {e}")
    else:
        print("[INFO] No existe CSV previo, se creará uno nuevo.")
    return codigos


def guardar_productos_csv(ruta_csv, productos_nuevos):
    if not productos_nuevos:
        return
    try:
        ruta_csv.parent.mkdir(parents=True, exist_ok=True)
        df_nuevos = pd.DataFrame(productos_nuevos)
        escribir_header = not ruta_csv.exists() or ruta_csv.stat().st_size == 0
        df_nuevos.to_csv(
            ruta_csv,
            mode="a",
            header=escribir_header,
            index=False,
            encoding="utf-8",
        )
        print(f"  [OK] {len(productos_nuevos)} productos guardados en {ruta_csv}")
    except Exception as e:
        print(f"[ERROR] Al guardar productos en CSV: {e}")


def consultar_productos_por_termino(token, termino, page_size=PAGE_SIZE):
    page_number = 0

    while True:
        try:
            r = requests.get(
                API_URL,
                headers=obtener_headers(token),
                params={
                    "q": termino,
                    "offer": "false",
                    "combo": "false",
                    "pageSize": page_size,
                    "pageNumber": page_number,
                },
                timeout=30,
            )
            r.raise_for_status()
            response_data = r.json()
        except requests.exceptions.RequestException as e:
            print(f"\n  [ERROR] Falló la petición para '{termino}' (pág {page_number}): {e}", flush=True)
            break
        except Exception as e:
            print(f"\n  [ERROR] Error inesperado en la consulta: {e}", flush=True)
            break

        if isinstance(response_data, dict):
            productos_pagina = response_data.get("product", [])
            total_pages = response_data.get("totalPages", 0)
        elif isinstance(response_data, list):
            productos_pagina = response_data
            total_pages = 1
        else:
            print(f"\n  [ADVERTENCIA] Respuesta inesperada para '{termino}': {type(response_data)}", flush=True)
            break

        if not productos_pagina:
            break

        yield page_number + 1, total_pages, productos_pagina
        page_number += 1

        if total_pages > 0 and page_number >= total_pages:
            break

        time.sleep(random.uniform(5.0, 12.0))

## 4. Ejecución del scraping
Orquesta la extracción de las 7 columnas estructurales por cada término y ejecuta el flujo principal.

In [ ]:
def ejecutar_scraping():
    if not API_URL or not TARGET_DOMAIN:
        raise ValueError("Faltan variables de entorno requeridas (API_URL, TARGET_DOMAIN). Revisa tu archivo .env")

    token = get_token()
    if not token:
        raise RuntimeError(
            "No se pudo obtener el token. Asegúrate de abrir Chrome con --remote-debugging-port=9222 "
            "y tener iniciada la sesión en el portal objetivo."
        )

    codigos_existentes = cargar_codigos_existentes_csv(OUTPUT_CSV)
    total_consultas = len(TERMINOS_BUSQUEDA)
    total_nuevos_global = 0

    print(f"\n[INICIO] Se procesarán {total_consultas} términos de búsqueda.\n", flush=True)

    for idx, termino in enumerate(TERMINOS_BUSQUEDA, 1):
        print(f"[{idx}/{total_consultas}] Buscando: '{termino}'...", flush=True)
        encontrados_termino = 0
        nuevos_termino = 0

        for pag, total_pags, productos_pagina in consultar_productos_por_termino(token, termino):
            encontrados_termino += len(productos_pagina)
            productos_nuevos_pagina = []

            for producto in productos_pagina:
                codigo = str(producto.get("barcode") or "").strip()
                if not codigo or codigo in codigos_existentes:
                    continue

                producto_filtrado = {
                    "codigo": codigo,
                    "producto": producto.get("presentation") or None,
                    "droga": normalizar_texto(producto.get("drug", "")),
                    "tipo": producto.get("type") or None,
                    "accion": normalizar_texto(producto.get("action", "")),
                    "clasificacion": normalizar_texto(producto.get("shape", "")),
                    "laboratorio": producto.get("laboratory") or None,
                }

                codigos_existentes.add(codigo)
                productos_nuevos_pagina.append(producto_filtrado)
                nuevos_termino += 1
                total_nuevos_global += 1

            total_str = total_pags if total_pags else "?"
            print(f"  -> pág {pag}/{total_str}: {len(productos_pagina)} recibidos, {len(productos_nuevos_pagina)} nuevos", flush=True)
            if productos_nuevos_pagina:
                guardar_productos_csv(OUTPUT_CSV, productos_nuevos_pagina)

        print(f"  [RESUMEN '{termino}'] {encontrados_termino} encontrados, {nuevos_termino} nuevos guardados.", flush=True)
        time.sleep(random.uniform(10.0, 20.0))

    print("\n[FIN] Proceso completado.")
    print(f"[INFO] Total de productos únicos registrados: {len(codigos_existentes):,}")
    print(f"[INFO] Productos nuevos agregados en esta ejecución: {total_nuevos_global:,}")
    print(f"[INFO] Archivo CSV guardado en: {OUTPUT_CSV.resolve()}")


ejecutar_scraping()